# Per-protein similarity statistics

Analyze `protein_neighbors_5013.json` — for each protein, the dict of other PDB IDs whose sequence identity is ≥ 30%.

In [3]:
import json
from collections import Counter
from statistics import mean, median, stdev

JSON_PATH = "/data/atran16/ProteinClassification_3D/3D_PDB_5013/protein_neighbors_5013_20percent.json"

with open(JSON_PATH) as f:
    data = json.load(f)

print(f"Loaded: {JSON_PATH}")
print(f"Total protein entries: {len(data):,}")

JSONDecodeError: Expecting ':' delimiter: line 4833861 column 11 (char 91307261)

In [ ]:
# ---------- Per-protein neighbor counts ----------
counts = {pid: len(nbrs) for pid, nbrs in data.items()}
total_proteins = len(counts)

with_nbrs    = [pid for pid, c in counts.items() if c > 0]
without_nbrs = [pid for pid, c in counts.items() if c == 0]

print(f"Proteins with    >=1 neighbor : {len(with_nbrs):,}  ({len(with_nbrs)/total_proteins:.1%})")
print(f"Proteins with NO   neighbor   : {len(without_nbrs):,}  ({len(without_nbrs)/total_proteins:.1%})")

print(f"\nMin neighbors : {min(counts.values())}")
print(f"Max neighbors : {max(counts.values())}")
print(f"Mean neighbors: {mean(counts.values()):.2f}")
print(f"Median        : {median(counts.values())}")
print(f"Stdev         : {stdev(counts.values()):.2f}")

Proteins with    >=1 neighbor : 4,838  (96.5%)
Proteins with NO   neighbor   : 175  (3.5%)

Min neighbors : 0
Max neighbors : 498
Mean neighbors: 99.42
Median        : 24
Stdev         : 156.17


In [ ]:
# ---------- Top / Bottom by neighbor count ----------
sorted_by_count = sorted(counts.items(), key=lambda x: (-x[1], x[0]))

print("Top 20 proteins with the MOST neighbors:")
for pid, c in sorted_by_count[:20]:
    print(f"  {pid}: {c:,} neighbors")

# bottom-most: only those with at least 1 neighbor (0-neighbor count is in the prev cell)
nonzero_sorted = sorted(((p, c) for p, c in counts.items() if c > 0),
                       key=lambda x: (x[1], x[0]))

print("\nTop 20 proteins with the FEWEST (>=1) neighbors:")
for pid, c in nonzero_sorted[:20]:
    print(f"  {pid}: {c} neighbors")

Top 20 proteins with the MOST neighbors:
  9QXL: 498 neighbors
  8DWH: 494 neighbors
  9IXX: 491 neighbors
  8K3Z: 489 neighbors
  9IZC: 489 neighbors
  8XZG: 488 neighbors
  9V0X: 488 neighbors
  8G05: 487 neighbors
  8IJB: 487 neighbors
  8IJD: 487 neighbors
  8Y52: 487 neighbors
  9GG5: 487 neighbors
  9JKY: 487 neighbors
  9KVG: 487 neighbors
  7XBW: 486 neighbors
  8GDC: 486 neighbors
  8IJ3: 486 neighbors
  8ZMD: 486 neighbors
  9M1H: 486 neighbors
  7VV3: 485 neighbors

Top 20 proteins with the FEWEST (>=1) neighbors:
  1R8X: 1 neighbors
  1R8Y: 1 neighbors
  1XFB: 1 neighbors
  2LZJ: 1 neighbors
  2V31: 1 neighbors
  4NOB: 1 neighbors
  4NOF: 1 neighbors
  4P58: 1 neighbors
  4P7F: 1 neighbors
  4YMK: 1 neighbors
  5IKO: 1 neighbors
  5YQG: 1 neighbors
  6BLY: 1 neighbors
  6HCY: 1 neighbors
  6LNI: 1 neighbors
  6RO4: 1 neighbors
  6RQJ: 1 neighbors
  6S7O: 1 neighbors
  6S7T: 1 neighbors
  6URG: 1 neighbors


In [ ]:
# ---------- Distribution of neighbor counts ----------
buckets = [(0, 0), (1, 1), (2, 5), (6, 10), (11, 25),
           (26, 50), (51, 100), (101, 250), (251, 500),
           (501, 1000), (1001, 5000)]

print("Neighbor-count distribution (how many proteins fall in each bucket):")
print(f"  {'range':>12}  {'#proteins':>10}   {'pct':>6}")
for lo, hi in buckets:
    cnt = sum(1 for c in counts.values() if lo <= c <= hi)
    label = f"{lo}" if lo == hi else f"{lo}-{hi}"
    print(f"  {label:>12}  {cnt:>10,}   {cnt/total_proteins:>6.1%}")

Neighbor-count distribution (how many proteins fall in each bucket):
         range   #proteins      pct
             0         175     3.5%
             1         214     4.3%
           2-5         759    15.1%
          6-10         481     9.6%
         11-25         960    19.2%
         26-50         944    18.8%
        51-100         324     6.5%
       101-250         259     5.2%
       251-500         897    17.9%
      501-1000           0     0.0%
     1001-5000           0     0.0%


In [ ]:
# ---------- All similarity scores (unordered pairs) ----------
seen_pairs = set()
all_sims = []
self_loops = 0

for pid, nbrs in data.items():
    for other, sim in nbrs.items():
        if pid == other:
            self_loops += 1
            continue
        key = (pid, other) if pid < other else (other, pid)
        if key in seen_pairs:
            continue
        seen_pairs.add(key)
        all_sims.append(sim)

print(f"Unique pairs (undirected) above threshold : {len(all_sims):,}")
print(f"Self-loop entries (should be 0)           : {self_loops}")

if all_sims:
    print(f"\nSimilarity score statistics (>= 30%):")
    print(f"  min    : {min(all_sims):.2f}")
    print(f"  max    : {max(all_sims):.2f}")
    print(f"  mean   : {mean(all_sims):.2f}")
    print(f"  median : {median(all_sims):.2f}")
    print(f"  stdev  : {stdev(all_sims):.2f}")

Unique pairs (undirected) above threshold : 249,203
Self-loop entries (should be 0)           : 0

Similarity score statistics (>= 30%):
  min    : 30.00
  max    : 100.00
  mean   : 55.23
  median : 49.56
  stdev  : 19.76


In [ ]:
# ---------- Similarity-score buckets ----------
sim_buckets = [(30, 40), (40, 50), (50, 60), (60, 70),
               (70, 80), (80, 90), (90, 99.99), (100, 100)]

print("Similarity score distribution (unique pairs):")
print(f"  {'range %':>10}  {'#pairs':>10}   {'pct':>6}")
for lo, hi in sim_buckets:
    cnt = sum(1 for s in all_sims if lo <= s <= hi)
    label = f"{lo:g}-{hi:g}" if lo != hi else f"{lo:g}"
    print(f"  {label:>10}  {cnt:>10,}   {cnt/len(all_sims):>6.1%}")

Similarity score distribution (unique pairs):
     range %      #pairs      pct
       30-40      70,899    28.5%
       40-50      55,708    22.4%
       50-60      38,219    15.3%
       60-70      30,507    12.2%
       70-80      21,638     8.7%
       80-90       9,135     3.7%
    90-99.99      12,502     5.0%
         100      10,875     4.4%


In [ ]:
# ---------- Top similarity pairs ----------
pair_with_sim = []
seen = set()
for pid, nbrs in data.items():
    for other, sim in nbrs.items():
        if pid == other:
            continue
        key = (pid, other) if pid < other else (other, pid)
        if key in seen:
            continue
        seen.add(key)
        pair_with_sim.append((sim, key[0], key[1]))

pair_with_sim.sort(reverse=True)

print("Top 20 pairs by similarity:")
for sim, a, b in pair_with_sim[:20]:
    print(f"  {a} <-> {b} : {sim:.2f}%")

print("\nBottom 20 pairs (closest to threshold):")
for sim, a, b in pair_with_sim[-20:]:
    print(f"  {a} <-> {b} : {sim:.2f}%")

Top 20 pairs by similarity:
  9ZQ0 <-> 9ZQ1 : 100.00%
  9ZPZ <-> 9ZQ1 : 100.00%
  9ZPZ <-> 9ZQ0 : 100.00%
  9ZPY <-> 9ZQ1 : 100.00%
  9ZPY <-> 9ZQ0 : 100.00%
  9ZPY <-> 9ZPZ : 100.00%
  9ZPX <-> 9ZQ1 : 100.00%
  9ZPX <-> 9ZQ0 : 100.00%
  9ZPX <-> 9ZPZ : 100.00%
  9ZPX <-> 9ZPY : 100.00%
  9ZPW <-> 9ZQ1 : 100.00%
  9ZPW <-> 9ZQ0 : 100.00%
  9ZPW <-> 9ZPZ : 100.00%
  9ZPW <-> 9ZPY : 100.00%
  9ZPW <-> 9ZPX : 100.00%
  9ZPV <-> 9ZQ1 : 100.00%
  9ZPV <-> 9ZQ0 : 100.00%
  9ZPV <-> 9ZPZ : 100.00%
  9ZPV <-> 9ZPY : 100.00%
  9ZPV <-> 9ZPX : 100.00%

Bottom 20 pairs (closest to threshold):
  7V83 <-> 8U4P : 30.00%
  7V83 <-> 8U4N : 30.00%
  7V82 <-> 8U4P : 30.00%
  7V82 <-> 8U4N : 30.00%
  7UXC <-> 8RCN : 30.00%
  7TL0 <-> 9IB3 : 30.00%
  7TL0 <-> 9IB1 : 30.00%
  7SBW <-> 8HGL : 30.00%
  7SBV <-> 8DLS : 30.00%
  7RMG <-> 8HSC : 30.00%
  7LUE <-> 8CIM : 30.00%
  7KZZ <-> 7XDL : 30.00%
  7KOO <-> 8X5X : 30.00%
  7JVC <-> 8WE4 : 30.00%
  7E9G <-> 9IVG : 30.00%
  6XZU <-> 8HN1 : 30.00%
  6V4O <-> 

In [ ]:
# ---------- Symmetry / consistency check ----------
asymmetric = []
disagreements = []

for pid, nbrs in data.items():
    for other, sim in nbrs.items():
        if pid == other:
            continue
        back = data.get(other, {}).get(pid)
        if back is None:
            asymmetric.append((pid, other, sim))
        elif abs(back - sim) > 0.01:
            disagreements.append((pid, other, sim, back))

print(f"Asymmetric edges (A->B exists but B->A missing): {len(asymmetric):,}")
if asymmetric[:10]:
    for pid, other, sim in asymmetric[:10]:
        print(f"  {pid} -> {other} ({sim:.2f}%)  but {other} has no entry for {pid}")

print(f"\nMismatched scores (A->B != B->A): {len(disagreements):,}")
for pid, other, s1, s2 in disagreements[:10]:
    print(f"  {pid}->{other}={s1:.2f}  vs  {other}->{pid}={s2:.2f}")

Asymmetric edges (A->B exists but B->A missing): 0

Mismatched scores (A->B != B->A): 0


In [ ]:
# ---------- Spot-check a few proteins ----------
sample_pids = [pid for pid, _ in sorted_by_count[:3]]               # busiest
sample_pids += [pid for pid, _ in nonzero_sorted[:3] if nonzero_sorted]  # sparsest non-zero

for pid in sample_pids:
    nbrs = data[pid]
    print(f"\n--- {pid} ({len(nbrs)} neighbors) ---")
    top10 = sorted(nbrs.items(), key=lambda x: -x[1])[:10]
    for other, sim in top10:
        print(f"   {other} : {sim:.2f}%")


--- 9QXL (498 neighbors) ---
   9GG5 : 58.10%
   7D7M : 55.87%
   9EEA : 53.88%
   9EE9 : 53.88%
   9JFT : 52.93%
   9M0Z : 52.71%
   9M1P : 52.71%
   9M3Q : 52.58%
   9M2S : 52.58%
   8HJ0 : 52.53%

--- 8DWH (494 neighbors) ---
   9JFV : 68.51%
   9BKK : 68.51%
   8WST : 67.31%
   9JCL : 64.77%
   8W89 : 64.75%
   8W8A : 64.75%
   8IQ6 : 64.07%
   8HDO : 63.51%
   8HDP : 63.51%
   9K0K : 63.41%

--- 9IXX (491 neighbors) ---
   9V0X : 77.39%
   8ZPT : 76.86%
   9V1H : 75.05%
   7YDJ : 71.31%
   7W55 : 70.20%
   7W57 : 69.85%
   9KVG : 68.49%
   8JEI : 66.22%
   8IYS : 66.17%
   9ITE : 65.36%

--- 1R8X (1 neighbors) ---
   1R8Y : 100.00%

--- 1R8Y (1 neighbors) ---
   1R8X : 100.00%

--- 1XFB (1 neighbors) ---
   8D44 : 70.14%
